# Scalability Benchmark

Measures runtime scaling with data dimensions.

- **Runtime vs rows**: Fixed N_cols=10, vary rows from 50 to 2000
- **Runtime vs columns**: Fixed N_rows=200, vary columns from 5 to 200
- **Per-sweep amortization**: lax.scan benefit as n_sweeps increases
- **JIT compilation time**: Overhead vs problem size

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/kaggle/working/jaxcross"
BRANCH = "chore/benchmark-consolidation"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && (git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}) && git pull origin {BRANCH}

%pip install -e . --no-deps -q

print(f"Branch: {BRANCH}")
print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

In [ ]:
import json
import time

import jax
import jax.numpy as jnp
import numpy as np

import crosscat.packed.state as _ps
from benchmarks.utils import create_results_dir, detect_platform, make_benchmark_data
from crosscat import initialize, pack_state, packed_gibbs_sweep
from crosscat.packed import batch_packed_states

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"
n_devices = jax.device_count()
print(f"Devices: {n_devices}x {jax.devices()}")


## 2. Runtime vs Number of Rows

Fixed N_cols=10, vary rows from 50 to 2000.

In [ ]:
def time_sweep(key, data, col_types, n_sweeps=5, n_warmup=1):
    """Time packed_gibbs_sweep: (compile_time, per_sweep_time)."""
    k1, k2, k3 = jax.random.split(key, 3)
    state = initialize(k1, data, col_types).state
    packed = pack_state(state)
    t0 = time.perf_counter()
    packed_w = packed_gibbs_sweep(k2, packed, data, n_sweeps=n_warmup)
    packed_w.column_assignments.block_until_ready()
    compile_time = time.perf_counter() - t0
    t0 = time.perf_counter()
    packed_out = packed_gibbs_sweep(k3, packed_w, data, n_sweeps=n_sweeps)
    packed_out.column_assignments.block_until_ready()
    total_time = time.perf_counter() - t0
    return compile_time, total_time / n_sweeps


base_key = jax.random.key(42)
row_counts = [50, 100, 200, 500, 1000, 2000]
n_cols = 10
rows_results = []

print("=== Scalability vs N_rows (N_cols=10) ===")
for n_rows in row_counts:
    key = jax.random.fold_in(base_key, n_rows)
    data, col_types = make_benchmark_data(key, n_rows, n_cols)
    k_time = jax.random.fold_in(key, 999)
    compile_t, sweep_t = time_sweep(k_time, data, col_types, n_sweeps=5)
    print(f"  N_rows={n_rows:5d}: compile={compile_t:.2f}s, sweep={sweep_t:.4f}s")
    rows_results.append(
        {
            "n_rows": n_rows,
            "n_cols": n_cols,
            "compile_time": compile_t,
            "per_sweep_time": sweep_t,
        }
    )

## 3. Runtime vs Number of Columns

Fixed N_rows=200, vary columns from 5 to 200.

In [ ]:
col_counts = [5, 10, 20, 50, 100, 200, 500, 1000]
n_rows = 200
cols_results = []

print("=== Scalability vs N_cols (N_rows=200) ===")
for n_cols in col_counts:
    key = jax.random.fold_in(base_key, n_cols + 10000)
    data, col_types = make_benchmark_data(key, n_rows, n_cols)
    k_time = jax.random.fold_in(key, 999)
    compile_t, sweep_t = time_sweep(k_time, data, col_types, n_sweeps=5)
    print(f"  N_cols={n_cols:5d}: compile={compile_t:.2f}s, sweep={sweep_t:.4f}s")
    cols_results.append(
        {
            "n_rows": n_rows,
            "n_cols": n_cols,
            "compile_time": compile_t,
            "per_sweep_time": sweep_t,
        }
    )

## 4. Per-Sweep Amortization

lax.scan amortization: per-sweep time drops as n_sweeps increases.

In [ ]:
sweep_counts = [1, 5, 10, 50, 100]
n_rows, n_cols = 200, 10
sweeps_results = []

print("=== Per-sweep time vs N_sweeps (200x10) ===")
key = jax.random.fold_in(base_key, 77777)
data, col_types = make_benchmark_data(key, n_rows, n_cols)
k_init = jax.random.fold_in(key, 0)
state = initialize(k_init, data, col_types).state
packed = pack_state(state)
k_warmup = jax.random.fold_in(key, 1)
packed = packed_gibbs_sweep(k_warmup, packed, data, n_sweeps=1)
packed.column_assignments.block_until_ready()

for n_sw in sweep_counts:
    k_run = jax.random.fold_in(key, n_sw)
    t0 = time.perf_counter()
    out = packed_gibbs_sweep(k_run, packed, data, n_sweeps=n_sw)
    out.column_assignments.block_until_ready()
    total = time.perf_counter() - t0
    per_sweep = total / n_sw
    print(f"  N_sweeps={n_sw:5d}: total={total:.3f}s, per_sweep={per_sweep:.4f}s")
    sweeps_results.append({"n_sweeps": n_sw, "total_time": total, "per_sweep_time": per_sweep})

## 5. Multi-Chain pmap Throughput

Compare sequential (single-GPU) vs pmap (multi-GPU) for running multiple chains.
Shows the speedup from distributing independent chains across 2xT4 GPUs.

In [ ]:
# Define pmap sweep function
def _sweep_one_chain(key, packed, data, n_sweeps):
    return packed_gibbs_sweep(key, packed, data, n_sweeps=n_sweeps)


def _sweep_chains_on_device(keys, packed_batch, data, n_sweeps):
    def body(i, carry):
        packed_b = carry
        single_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)[i]
        for name in _ps._STATIC_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)
        single = _ps.PackedCrossCatState(**single_kwargs)
        result = _sweep_one_chain(keys[i], single, data, n_sweeps)
        new_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            arr = getattr(packed_b, name)
            new_kwargs[name] = arr.at[i].set(getattr(result, name))
        for name in _ps._STATIC_FIELDS:
            new_kwargs[name] = getattr(packed_b, name)
        return _ps.PackedCrossCatState(**new_kwargs)
    return jax.lax.fori_loop(0, keys.shape[0], body, packed_batch)


pmap_sweep = jax.pmap(
    _sweep_chains_on_device,
    in_axes=(0, 0, None, None),
    static_broadcasted_argnums=(3,),
)

# Benchmark: 4 chains x 10 sweeps on 500 rows x 20 cols
N_BENCH_CHAINS = max(4, n_devices * 2)
CHAINS_PER_DEVICE = N_BENCH_CHAINS // n_devices
N_BENCH_CHAINS = CHAINS_PER_DEVICE * n_devices
n_bench_sweeps = 10
n_bench_rows, n_bench_cols = 500, 20

print(f"=== pmap Multi-Chain: {N_BENCH_CHAINS} chains x {n_bench_sweeps} sweeps ({n_bench_rows}x{n_bench_cols}) ===")
print(f"    {CHAINS_PER_DEVICE} chains/device, {n_devices} devices")

k_bench = jax.random.key(88888)
bench_data, bench_col_types = make_benchmark_data(k_bench, n_bench_rows, n_bench_cols)

# Initialize chains
init_keys = jax.random.split(jax.random.fold_in(k_bench, 1), N_BENCH_CHAINS)
bench_packed = []
for c in range(N_BENCH_CHAINS):
    s = initialize(init_keys[c], bench_data, bench_col_types).state
    bench_packed.append(pack_state(s))

# --- Sequential baseline (single GPU) ---
seq_keys = jax.random.split(jax.random.fold_in(k_bench, 2), N_BENCH_CHAINS)

# Warmup
for c in range(N_BENCH_CHAINS):
    bench_packed[c] = packed_gibbs_sweep(seq_keys[c], bench_packed[c], bench_data, n_sweeps=1)
    bench_packed[c].column_assignments.block_until_ready()

seq_keys = jax.random.split(jax.random.fold_in(k_bench, 3), N_BENCH_CHAINS)
t0 = time.perf_counter()
for c in range(N_BENCH_CHAINS):
    bench_packed[c] = packed_gibbs_sweep(seq_keys[c], bench_packed[c], bench_data, n_sweeps=n_bench_sweeps)
    bench_packed[c].column_assignments.block_until_ready()
seq_time = time.perf_counter() - t0
print(f"  Sequential: {seq_time:.2f}s ({seq_time / N_BENCH_CHAINS:.2f}s/chain)")

# --- pmap (multi-GPU) ---
pmap_keys = jax.random.split(jax.random.fold_in(k_bench, 4), N_BENCH_CHAINS)

# Warmup pmap
batched = batch_packed_states(bench_packed)
keys_pmap = pmap_keys.reshape(n_devices, CHAINS_PER_DEVICE, *pmap_keys.shape[1:])
bkw = {}
for name in _ps._ARRAY_FIELDS:
    arr = getattr(batched, name)
    bkw[name] = arr.reshape((n_devices, CHAINS_PER_DEVICE) + arr.shape[1:])
for name in _ps._STATIC_FIELDS:
    bkw[name] = getattr(batched, name)
batched_pmap = _ps.PackedCrossCatState(**bkw)
_ = pmap_sweep(keys_pmap, batched_pmap, bench_data, 1)
jax.tree.map(lambda x: x.block_until_ready(), _)

# Timed run
pmap_keys = jax.random.split(jax.random.fold_in(k_bench, 5), N_BENCH_CHAINS)
keys_pmap = pmap_keys.reshape(n_devices, CHAINS_PER_DEVICE, *pmap_keys.shape[1:])
t0 = time.perf_counter()
result_pmap = pmap_sweep(keys_pmap, batched_pmap, bench_data, n_bench_sweeps)
jax.tree.map(lambda x: x.block_until_ready(), result_pmap)
pmap_time = time.perf_counter() - t0
speedup = seq_time / max(pmap_time, 1e-9)
print(f"  pmap:       {pmap_time:.2f}s ({pmap_time / N_BENCH_CHAINS:.2f}s/chain)")
print(f"  Speedup:    {speedup:.1f}x")

pmap_results = {
    "n_chains": N_BENCH_CHAINS,
    "chains_per_device": CHAINS_PER_DEVICE,
    "n_devices": n_devices,
    "n_sweeps": n_bench_sweeps,
    "n_rows": n_bench_rows,
    "n_cols": n_bench_cols,
    "sequential_time": seq_time,
    "pmap_time": pmap_time,
    "speedup": speedup,
}

## 6. Scalability Plots

In [ ]:
import matplotlib.pyplot as plt

results_dir = create_results_dir("scalability")
all_results = {
    "vs_rows": rows_results,
    "vs_cols": cols_results,
    "vs_sweeps": sweeps_results,
    "backend": platform["backend"],
    "device": str(jax.devices()[0]),
    "pmap_benchmark": pmap_results,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (a) rows
ax = axes[0]
xs = [r["n_rows"] for r in rows_results]
ys = [r["per_sweep_time"] for r in rows_results]
ax.plot(xs, ys, "o-", color="#2196F3", linewidth=2, markersize=6)
ax.set_xlabel("Number of rows")
ax.set_ylabel("Time per sweep (s)")
ax.set_title("(a) Scaling with rows")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# (b) cols
ax = axes[1]
xs = [r["n_cols"] for r in cols_results]
ys = [r["per_sweep_time"] for r in cols_results]
ax.plot(xs, ys, "s-", color="#4CAF50", linewidth=2, markersize=6)
ax.set_xlabel("Number of columns")
ax.set_ylabel("Time per sweep (s)")
ax.set_title("(b) Scaling with columns")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# (c) compile
ax = axes[2]
sizes_r = [r["n_rows"] * r["n_cols"] for r in rows_results]
compile_r = [r["compile_time"] for r in rows_results]
sizes_c = [r["n_rows"] * r["n_cols"] for r in cols_results]
compile_c = [r["compile_time"] for r in cols_results]
ax.scatter(sizes_r, compile_r, marker="o", color="#2196F3", label="Vary rows", s=40)
ax.scatter(sizes_c, compile_c, marker="s", color="#4CAF50", label="Vary cols", s=40)
ax.set_xlabel("Table size (rows x cols)")
ax.set_ylabel("JIT compile time (s)")
ax.set_title("(c) Compilation overhead")
ax.set_xscale("log")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(results_dir / "scalability.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Save Results

In [ ]:
import shutil


# Save JSON
def convert(obj):
    """Convert numpy/jax types for JSON serialization."""
    if isinstance(obj, (np.integer, jnp.integer)):
        return int(obj)
    if isinstance(obj, (np.floating, jnp.floating, float)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


with open(results_dir / "scalability_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=convert)

print(f"Results saved to {results_dir}")

# Archive to /kaggle/working/ for easy download
results_tar = "/kaggle/working/scalability_results.tar.gz"
shutil.make_archive("/kaggle/working/scalability_results", "gztar", ".", str(results_dir))
print(f"Archived to {results_tar}")
print("Download from Kaggle Output tab.")